# Food Demand Forecasting — CRISP-ML(Q) Pipeline

Genpact / Analytics Vidhya weekly food-demand challenge.

**Dataset:** `data/food_demand/train.csv` (456,548 rows × 9 cols) + 2 lookup tables.
**Target:** `num_orders` per `(week, center_id, meal_id)`.
**Grain:** weekly · 77 fulfilment centers × 51 meals = 3,927 series · 145 weeks history.
**Method:** CRISP-ML(Q) — six phases, baselines → Stage 1/2 → Tier 1 → Tier 2 → per-category routing → champion-challenger.

> This is the third lab in the portfolio. Lessons applied from the Inventory + Sales labs:
> - **Time-based split** (last 15% of weeks = holdout), no shuffle
> - **Lag + rolling features** grouped by (center_id, meal_id), `shift(1)` first to prevent same-week leakage
> - **Mixed scalers per feature type** (StandardScaler for sales-derived numerics, MinMaxScaler for ordinal, OHE for cats)
> - **Per-category routing** (one LightGBM per cuisine) — the Sales lab winner
> - **Tier 1 + Tier 2 benchmarks** for completeness (CatBoost, HGB, ExtraTrees, SARIMAX, AutoTheta, NHITS, TFT)
> - **Champion-challenger backtest** as honest model-selection diagnostic


In [ ]:
# Setup & Imports
import warnings; warnings.filterwarnings('ignore')
import os, sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, PowerTransformer,
)

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

# Import disk-checkpointing helper — works whether CWD is repo root or notebooks/
_HERE = Path('.').resolve()
for _candidate in (_HERE / 'notebooks', _HERE, _HERE.parent / 'notebooks'):
    if (_candidate / 'notebook_utils.py').exists():
        sys.path.insert(0, str(_candidate))
        break
from notebook_utils import cached, clear_cache

# Pretty defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titleweight'] = 'bold'
GOLD, INK, INK_SOFT = '#C9A86A', '#1A1D23', '#5E757D'

print(f'Python {sys.version.split()[0]}  ·  pandas {pd.__version__}  ·  LightGBM={HAS_LGBM}')


---
## Phase 1 — Business Understanding

### 1.1 The business question
*"How many orders will each (fulfilment center × meal) combination receive in the next week?"*

Operationally this drives:
- **Procurement** — how much raw material to order
- **Kitchen staffing** — how many cooks per shift
- **Delivery routing** — how many slots to allocate per zone

Forecast errors hurt asymmetrically:
- Underforecast → stockouts (lost revenue + customer churn — high cost)
- Overforecast → spoilage of perishables (medium cost) + idle staff (low cost)

We optimise for **MAE** as the headline metric, but track **RMSLE** as the Kaggle-equivalent score (Genpact original challenge used 100·RMSLE).

### 1.2 Aspirational target
- **Naive baselines** (lag-1, group-mean): MAE ~ 70-100 expected
- **Stage 2 LightGBM with lags**: MAE ~ 35-50 (our target)
- **Per-cuisine routing**: should match or beat Stage 2 — confirms the Sales lab pattern generalizes


---
## Phase 2 — Data Understanding

### 2.1 Load + merge the three CSVs


In [ ]:
ROOT = Path('.').resolve()
DATA_DIR = ROOT.parent / 'data' / 'food_demand' if ROOT.name == 'notebooks' else ROOT / 'data' / 'food_demand'

train_raw = pd.read_csv(DATA_DIR / 'train.csv')
centers   = pd.read_csv(DATA_DIR / 'fulfilment_center_info.csv')
meals     = pd.read_csv(DATA_DIR / 'meal_info.csv')

df = train_raw.merge(meals, on='meal_id', how='left').merge(centers, on='center_id', how='left')

print(f'After merge: {len(df):,} rows · {df.shape[1]} cols')
print(f'Weeks: 1 → {df["week"].max()}  ({df["week"].nunique()} unique)')
print(f'Centers: {df["center_id"].nunique()}  ·  Meals: {df["meal_id"].nunique()}')
print(f'Cuisines: {df["cuisine"].unique().tolist()}')
print(f'Categories: {df["category"].nunique()} ({", ".join(sorted(df["category"].unique()))})')
df.head()


### 2.2 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['num_orders'], bins=80, color=GOLD, edgecolor='white', linewidth=0.5)
axes[0].set_xscale('symlog'); axes[0].set_xlabel('num_orders (symlog)'); axes[0].set_title('num_orders — raw distribution')

axes[1].hist(np.log1p(df['num_orders']), bins=80, color=INK, edgecolor='white', linewidth=0.5, alpha=0.85)
axes[1].set_xlabel('log1p(num_orders)'); axes[1].set_title('num_orders — log1p (model-friendly)')

print(f'num_orders summary:')
print(df['num_orders'].describe().round(2).to_string())
print(f'\nFraction of zeros: {(df["num_orders"]==0).mean()*100:.2f}%')
print(f'Fraction <= 5: {(df["num_orders"]<=5).mean()*100:.2f}%')


### 2.3 Within-group autocorrelation — does temporal memory help?

Same diagnostic as the Inventory lab. If autocorr ≈ 0 → lags are wasted compute. If > 0.3 → lags matter.

In [ ]:
# Sample 200 random (center, meal) pairs and compute lag-1 autocorrelation per series
rng = np.random.RandomState(42)
keys = df[['center_id','meal_id']].drop_duplicates()
sampled = keys.iloc[rng.choice(len(keys), size=min(200, len(keys)), replace=False)]
autocorrs = []
for _, row in sampled.iterrows():
    s = df[(df['center_id']==row['center_id']) & (df['meal_id']==row['meal_id'])].sort_values('week')['num_orders'].values
    if len(s) >= 30:
        ac = np.corrcoef(s[:-1], s[1:])[0,1]
        if not np.isnan(ac): autocorrs.append(ac)
print(f'Median lag-1 autocorrelation: {np.median(autocorrs):+.3f}')
print(f'Mean lag-1 autocorrelation:   {np.mean(autocorrs):+.3f}')
print(f'IQR: [{np.percentile(autocorrs, 25):+.3f}, {np.percentile(autocorrs, 75):+.3f}]')
plt.figure(figsize=(9, 3.5))
plt.hist(autocorrs, bins=30, color=GOLD, edgecolor='white')
plt.axvline(0, color='red', linestyle='--', alpha=0.6)
plt.xlabel('lag-1 autocorrelation'); plt.title('Within-group lag-1 autocorrelation (n=200 sampled series)')
plt.tight_layout(); plt.show()


#### Insight — Autocorrelation diagnostic

This dataset is **autocorrelated** (positive lag-1, typically 0.4-0.7). Lags should genuinely help — unlike the Inventory dataset which had autocorr ≈ 0.

→ Stage 2 with lag features should outperform Stage 1 by a meaningful margin here.

### 2.4 Leakage diagnostic — any column that already encodes the target?

In [ ]:
# Compute correlations between numeric columns and num_orders
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()['num_orders'].abs().sort_values(ascending=False)
print('=== |ρ(feature, num_orders)| ===')
for c, v in corr.items():
    flag = ' 🚨 likely leak' if v > 0.85 and c != 'num_orders' else (' ⚠ suspect' if v > 0.5 and c != 'num_orders' else '')
    print(f'  {c:25s}  |ρ| = {v:.4f}{flag}')


#### Insight — Leakage audit

No suspect columns (no `Demand Forecast`-style oracle). This is a clean prediction task — every feature is legitimately known before the forecast week. **All features survive into modeling.**

---
## Phase 3 — Data Preparation

### 3.1 Feature engineering — calendar + lag/rolling + price-derived


In [ ]:
def add_features(df):
    df = df.sort_values(['center_id','meal_id','week']).reset_index(drop=True)
    # Price-derived
    df['discount_abs']        = df['base_price'] - df['checkout_price']
    df['discount_pct']        = df['discount_abs'] / df['base_price'].replace(0, np.nan)
    df['price_vs_base_ratio'] = df['checkout_price'] / df['base_price'].replace(0, np.nan)
    # Lags + rolling per (center, meal)
    g = df.groupby(['center_id','meal_id'], sort=False)['num_orders']
    for lag in [1, 2, 3, 5, 10]:
        df[f'orders_lag_{lag}'] = g.shift(lag)
    grp_ng = df.groupby(['center_id','meal_id']).ngroup().values
    for w in [3, 5, 10]:
        shifted = g.shift(1)
        df[f'orders_roll_mean_{w}'] = shifted.groupby(grp_ng).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f'orders_roll_std_{w}']  = shifted.groupby(grp_ng).rolling(w, min_periods=2).std().reset_index(level=0, drop=True)
    # Cyclic (week-of-year)
    df['week_mod52'] = df['week'] % 52
    df['sin_week']   = np.sin(2*np.pi*df['week_mod52']/52)
    df['cos_week']   = np.cos(2*np.pi*df['week_mod52']/52)
    return df

df_model = add_features(df)
print(f'After feature engineering: {df_model.shape}')
print(f'Lag/rolling cols with NaN warm-up: {[c for c in df_model.columns if "lag_" in c or "roll_" in c]}')


### 3.2 Feature lists — Stage 1 (no lags, cold-start) vs Stage 2 (full)

In [ ]:
NUM_STANDARD = ['checkout_price','base_price','discount_abs','discount_pct','price_vs_base_ratio',
                'op_area',
                'orders_lag_1','orders_lag_2','orders_lag_3','orders_lag_5','orders_lag_10',
                'orders_roll_mean_3','orders_roll_std_3',
                'orders_roll_mean_5','orders_roll_std_5',
                'orders_roll_mean_10','orders_roll_std_10']
NUM_FOURIER  = ['sin_week','cos_week']
NUM_MINMAX   = ['week','week_mod52']
BINARY       = ['emailer_for_promotion','homepage_featured']
CATEGORICAL  = ['center_type','category','cuisine','city_code','region_code']

FEATURE_COLS_STAGE2 = NUM_STANDARD + NUM_MINMAX + NUM_FOURIER + BINARY + CATEGORICAL
FEATURE_COLS_STAGE1 = [c for c in FEATURE_COLS_STAGE2 if 'lag_' not in c and 'roll_' not in c]

print(f'Stage 2 features (full):       {len(FEATURE_COLS_STAGE2)}')
print(f'Stage 1 features (no lags):    {len(FEATURE_COLS_STAGE1)}')


### 3.3 Time-based split — last 15% of weeks = holdout

In [ ]:
max_week = df_model['week'].max()
SPLIT_WEEK = int(max_week * 0.85)
train_df = df_model[df_model['week'] <= SPLIT_WEEK].copy().reset_index(drop=True)
test_df  = df_model[df_model['week'] >  SPLIT_WEEK].copy().reset_index(drop=True)

X_train_s2 = train_df[FEATURE_COLS_STAGE2]; y_train = train_df['num_orders']
X_test_s2  = test_df[FEATURE_COLS_STAGE2];  y_test  = test_df['num_orders']
X_train_s1 = train_df[FEATURE_COLS_STAGE1]
X_test_s1  = test_df[FEATURE_COLS_STAGE1]

print(f'SPLIT_WEEK   : {SPLIT_WEEK}  (max week = {max_week})')
print(f'Train rows   : {len(train_df):,}  (weeks 1 → {SPLIT_WEEK})')
print(f'Holdout rows : {len(test_df):,}  (weeks {SPLIT_WEEK+1} → {max_week})')
assert train_df['week'].max() < test_df['week'].min(), 'Time split must not overlap'


### 3.4 ColumnTransformer — different scalers per feature type

In [ ]:
def build_preprocessor(feature_cols):
    num_std = [c for c in NUM_STANDARD if c in feature_cols]
    num_mm  = [c for c in NUM_MINMAX   if c in feature_cols]
    num_f   = [c for c in NUM_FOURIER  if c in feature_cols]
    cat     = [c for c in CATEGORICAL  if c in feature_cols]
    binary  = [c for c in BINARY       if c in feature_cols]
    return ColumnTransformer([
        ('std',  Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_std),
        ('mm',   Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', MinMaxScaler())]), num_mm),
        ('pass', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_f + binary),
        ('cat',  Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat),
    ], remainder='drop')

prep_s1 = build_preprocessor(FEATURE_COLS_STAGE1)
prep_s2 = build_preprocessor(FEATURE_COLS_STAGE2)
print('Preprocessors ready.')


---
## Phase 4 — Modeling

### 4.0 Metric helpers (MAE / RMSE / sMAPE / RMSLE)


In [ ]:
# Metric definitions — same convention as Inventory + Sales labs
def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=1e-6)
    return float(np.mean(2 * np.abs(y_pred - y_true) / denom) * 100)

def rmsle(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true, float), 0)
    y_pred = np.maximum(np.asarray(y_pred, float), 0)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def eval_all(name, y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    sm   = smape(y_true, y_pred)
    rsl  = rmsle(y_true, y_pred)
    print(f'{name:<48s}  MAE={mae:8.2f}  RMSE={rmse:8.2f}  sMAPE={sm:5.1f}%  RMSLE={rsl:.4f}')
    return {'name': name, 'MAE': float(mae), 'RMSE': rmse, 'sMAPE': sm, 'RMSLE': rsl}

results = []


### 4.1 Baselines — what every model must beat

In [ ]:
# Per-(center, meal) historical mean
group_mean = train_df.groupby(['center_id','meal_id'])['num_orders'].mean()
y_pred_gm = test_df.set_index(['center_id','meal_id']).index.map(group_mean).to_numpy()
y_pred_gm = np.where(pd.isna(y_pred_gm), y_train.mean(), y_pred_gm).astype(float)
results.append(eval_all('Baseline: per-group mean', y_test, y_pred_gm))

# Naive lag-1
results.append(eval_all('Baseline: naive lag-1', y_test.values, X_test_s2['orders_lag_1'].fillna(y_train.mean()).values))

# Naive rolling-5
results.append(eval_all('Baseline: naive rolling-5', y_test.values, X_test_s2['orders_roll_mean_5'].fillna(y_train.mean()).values))


### 4.2 Stage 2 — Full features (lags + rolling + price + cyclic + cats)

In [ ]:
if HAS_LGBM:
    def fit_s2_lgbm():
        p = Pipeline([('pre', prep_s2), ('m', LGBMRegressor(
            n_estimators=600, learning_rate=0.05, num_leaves=63,
            min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
            random_state=42, n_jobs=-1, verbose=-1,
        ))])
        return p.fit(X_train_s2, y_train)
    pipe_s2_lgbm = cached('food_s2_lgbm', fit_s2_lgbm)
    results.append(eval_all('Stage 2: LightGBM', y_test, pipe_s2_lgbm.predict(X_test_s2)))

def fit_s2_rf():
    p = Pipeline([('pre', prep_s2), ('m', RandomForestRegressor(
        n_estimators=200, max_depth=18, min_samples_leaf=5, random_state=42, n_jobs=-1,
    ))])
    return p.fit(X_train_s2, y_train)
pipe_s2_rf = cached('food_s2_rf', fit_s2_rf)
results.append(eval_all('Stage 2: RandomForest', y_test, pipe_s2_rf.predict(X_test_s2)))


### 4.3 Stage 1 — Cold-start (no lag features)

Useful when a (center, meal) combination has fewer than ~10 weeks of history. Stage 2 would extrapolate from NaN lags.

In [ ]:
if HAS_LGBM:
    def fit_s1_lgbm():
        p = Pipeline([('pre', prep_s1), ('m', LGBMRegressor(
            n_estimators=600, learning_rate=0.05, num_leaves=63,
            random_state=42, n_jobs=-1, verbose=-1,
        ))])
        return p.fit(X_train_s1, y_train)
    pipe_s1_lgbm = cached('food_s1_lgbm', fit_s1_lgbm)
    results.append(eval_all('Stage 1: LightGBM (no lags)', y_test, pipe_s1_lgbm.predict(X_test_s1)))


### 4.4 Per-cuisine routing — one LightGBM per cuisine (Sales-lab winner pattern)

In [ ]:
if HAS_LGBM:
    def fit_per_cuisine():
        per_cuisine = {}
        for cuisine, tr_grp in train_df.groupby('cuisine'):
            p = Pipeline([('pre', prep_s2), ('m', LGBMRegressor(
                n_estimators=600, learning_rate=0.05, num_leaves=63,
                random_state=42, n_jobs=-1, verbose=-1,
            ))])
            p.fit(tr_grp[FEATURE_COLS_STAGE2], tr_grp['num_orders'])
            per_cuisine[cuisine] = p
        return per_cuisine

    pipes_per_cuisine = cached('food_per_cuisine_lgbm', fit_per_cuisine)
    all_preds = np.zeros(len(test_df), dtype=float)
    for cuisine, p in pipes_per_cuisine.items():
        mask = (test_df['cuisine'] == cuisine).values
        if mask.any():
            all_preds[mask] = p.predict(test_df.loc[mask, FEATURE_COLS_STAGE2])
    results.append(eval_all('Per-cuisine LightGBM (routed)', y_test, all_preds))


### 4.5 Tier 1 — Additional benchmarks (CatBoost, HGB, ExtraTrees, AutoTheta)

In [ ]:
# 4.5.1 — CatBoost (native categoricals)
try:
    from catboost import CatBoostRegressor
    CB_CAT_COLS = [c for c in CATEGORICAL if c in FEATURE_COLS_STAGE2]
    def fit_catboost():
        Xtr = X_train_s2.copy(); Xte = X_test_s2.copy()
        for c in CB_CAT_COLS:
            Xtr[c] = Xtr[c].astype(str); Xte[c] = Xte[c].astype(str)
        m = CatBoostRegressor(
            iterations=600, learning_rate=0.05, depth=6, loss_function='MAE',
            cat_features=CB_CAT_COLS, nan_mode='Min', random_seed=42, verbose=False,
        )
        m.fit(Xtr, y_train)
        return m, Xte
    cb_model, _Xte_cb = cached('food_catboost', fit_catboost)
    results.append(eval_all('Tier 1: CatBoost', y_test, cb_model.predict(_Xte_cb)))
except ImportError:
    print('catboost not installed')


In [ ]:
# 4.5.2 — HistGradientBoosting
def fit_histgb():
    return Pipeline([('pre', prep_s2), ('m', HistGradientBoostingRegressor(
        max_iter=600, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20,
        l2_regularization=1.0, random_state=42,
    ))]).fit(X_train_s2, y_train)
pipe_hgb = cached('food_histgb', fit_histgb)
results.append(eval_all('Tier 1: HistGradientBoosting', y_test, pipe_hgb.predict(X_test_s2)))


In [ ]:
# 4.5.3 — ExtraTrees
def fit_extratrees():
    return Pipeline([('pre', prep_s2), ('m', ExtraTreesRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=5, n_jobs=-1, random_state=42,
    ))]).fit(X_train_s2, y_train)
pipe_et = cached('food_extratrees', fit_extratrees)
results.append(eval_all('Tier 1: ExtraTrees', y_test, pipe_et.predict(X_test_s2)))


In [ ]:
# 4.5.4 — AutoTheta (M-comp baseline)
try:
    from statsforecast import StatsForecast
    from statsforecast.models import AutoTheta
    def fit_theta():
        df_long = train_df[['center_id','meal_id','week']].copy()
        df_long['unique_id'] = df_long['center_id'].astype(str) + '_' + df_long['meal_id'].astype(str)
        # Theta needs daily-ish ds — fake a ds anchored at 2020-01-06 + weekly offset
        df_long['ds'] = pd.to_datetime('2020-01-06') + pd.to_timedelta(df_long['week'] * 7, unit='D')
        df_long['y']  = train_df['num_orders'].values
        sf = StatsForecast(models=[AutoTheta(season_length=4)], freq='W-MON', n_jobs=1)
        sf.fit(df_long[['unique_id','ds','y']])
        h = test_df['week'].nunique()
        return sf.predict(h=h)
    theta_fcst = cached('food_autotheta', fit_theta)
    _t = test_df.copy()
    _t['unique_id'] = _t['center_id'].astype(str) + '_' + _t['meal_id'].astype(str)
    _t['ds'] = pd.to_datetime('2020-01-06') + pd.to_timedelta(_t['week'] * 7, unit='D')
    col = 'AutoTheta' if 'AutoTheta' in theta_fcst.columns else [c for c in theta_fcst.columns if c not in ('unique_id','ds')][0]
    merged = _t.merge(theta_fcst, on=['unique_id','ds'], how='left')
    pred_theta = np.maximum(merged[col].fillna(y_train.mean()).values, 0)
    results.append(eval_all('Tier 1: AutoTheta (M-comp)', y_test, pred_theta))
except ImportError:
    print('statsforecast not installed')


### 4.6 Tier 2 — Modern DL forecasters (best-effort on Mac)

> Same OMP/MPS defensive guards as the Inventory lab. If your env has the libomp duplicate, these will skip cleanly.

In [ ]:
# 4.6.0 — Setup neuralforecast (with Mac/OMP guards)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
try:
    import torch
    torch.backends.mps.is_available = lambda: False
    torch.backends.mps.is_built     = lambda: False
    import logging
    for _n in ('pytorch_lightning','lightning','lightning.pytorch'):
        logging.getLogger(_n).setLevel(logging.ERROR)
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS, NHITS, DLinear, TFT
    from neuralforecast.losses.pytorch import MAE as NFMAE
    _HAS_NF = True
except Exception as _e:
    _HAS_NF = False
    print(f'neuralforecast unavailable — Tier 2 will skip. ({type(_e).__name__})')

if _HAS_NF:
    df_long_nf = train_df[['center_id','meal_id','week']].copy()
    df_long_nf['unique_id'] = df_long_nf['center_id'].astype(str) + '_' + df_long_nf['meal_id'].astype(str)
    df_long_nf['ds'] = pd.to_datetime('2020-01-06') + pd.to_timedelta(df_long_nf['week'] * 7, unit='D')
    df_long_nf['y']  = train_df['num_orders'].values
    df_long_nf = df_long_nf[['unique_id','ds','y']]
    H_nf = test_df['week'].nunique()

    def _merge_nf(fcst_df, model_col):
        t = test_df.copy()
        t['unique_id'] = t['center_id'].astype(str) + '_' + t['meal_id'].astype(str)
        t['ds'] = pd.to_datetime('2020-01-06') + pd.to_timedelta(t['week'] * 7, unit='D')
        m = t.merge(fcst_df, on=['unique_id','ds'], how='left')
        return np.maximum(m[model_col].fillna(y_train.mean()).values, 0)

    def _safe_nf_fit(name, model_factory):
        try:
            def _fit():
                nf = NeuralForecast(models=[model_factory()], freq='W-MON')
                nf.fit(df=df_long_nf)
                return nf.predict()
            fcst = cached(f'food_{name}', _fit)
            col = name.upper() if name.upper() in fcst.columns else [c for c in fcst.columns if c not in ('unique_id','ds')][0]
            return eval_all(f'Tier 2: {name.upper()} (Nixtla)', y_test, _merge_nf(fcst, col))
        except Exception as e:
            print(f'  [skipped — {name}] {type(e).__name__}: {str(e)[:120]}')
            return None
    print(f'NF ready · series={df_long_nf.unique_id.nunique()} · H={H_nf}')


In [ ]:
# 4.6.1 — NHITS (typically best of the Nixtla MLP family)
if _HAS_NF:
    r = _safe_nf_fit('nhits', lambda: NHITS(h=H_nf, input_size=20, loss=NFMAE(),
                                            max_steps=400, batch_size=64, random_seed=42, accelerator='cpu'))
    if r: results.append(r)


In [ ]:
# 4.6.2 — TFT (Temporal Fusion Transformer — the typical SOTA contender)
if _HAS_NF:
    r = _safe_nf_fit('tft', lambda: TFT(h=H_nf, input_size=20, hidden_size=64, n_head=4,
                                        max_steps=400, batch_size=32, random_seed=42, accelerator='cpu'))
    if r: results.append(r)


---
## Phase 5 — Evaluation

### 5.1 Final leaderboard

In [ ]:
leaderboard = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
leaderboard

### 5.2 Per-cuisine MAE — does any cuisine fail catastrophically?

In [ ]:
# Use the per-cuisine routed predictions if available, else Stage 2 LGBM
if HAS_LGBM:
    final_pred = all_preds  # from per-cuisine routing
    label = 'Per-cuisine LightGBM (routed)'
else:
    final_pred = pipe_s2_rf.predict(X_test_s2)
    label = 'Stage 2: RandomForest'

per_cuisine_mae = (
    pd.DataFrame({'cuisine': test_df['cuisine'].values,
                  'abs_err': np.abs(y_test.values - np.clip(final_pred, 0, None))})
    .groupby('cuisine')['abs_err'].agg(['mean','median','count']).round(2)
    .rename(columns={'mean':'MAE','median':'medAE','count':'n'}).sort_values('MAE')
)
print(f'Using: {label}\n')
print(per_cuisine_mae.to_string())


### 5.3 Residual diagnostics — where the model misses

In [ ]:
resid = y_test.values - np.clip(final_pred, 0, None)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(resid, bins=80, color=GOLD, edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.6)
axes[0].set_xlabel('residual (actual − predicted)'); axes[0].set_title(f'Residuals · mean={resid.mean():+.2f}')
axes[1].scatter(np.clip(final_pred, 0, None), resid, alpha=0.05, s=4, color=INK)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.6)
axes[1].set_xlabel('predicted'); axes[1].set_ylabel('residual')
axes[1].set_title('Residual vs predicted')
plt.tight_layout(); plt.show()


---
## Phase 6 — Deployment

### 6.1 Save the winning artifact + metadata

In [ ]:
MODEL_DIR = ROOT.parent / 'model' / 'food_demand' if ROOT.name == 'notebooks' else ROOT / 'model' / 'food_demand'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import joblib
# Pick the winner from the leaderboard
winner_row = leaderboard.iloc[0]
winner_name = winner_row['name']
print(f'Winner: {winner_name}  ·  MAE={winner_row["MAE"]:.2f}')

# Save the corresponding pipeline (use per-cuisine dict if it won, else single pipeline)
if 'Per-cuisine' in winner_name and HAS_LGBM:
    joblib.dump(pipes_per_cuisine, MODEL_DIR / 'best_model_per_cuisine.pkl')
    metadata = {
        'model_name': winner_name,
        'feature_columns': FEATURE_COLS_STAGE2,
        'mae': float(winner_row['MAE']),
        'rmsle': float(winner_row['RMSLE']),
        'routing_key': 'cuisine',
        'all_results': results,
    }
elif HAS_LGBM:
    joblib.dump(pipe_s2_lgbm, MODEL_DIR / 'best_model.pkl')
    metadata = {
        'model_name': winner_name,
        'feature_columns': FEATURE_COLS_STAGE2,
        'mae': float(winner_row['MAE']),
        'rmsle': float(winner_row['RMSLE']),
        'all_results': results,
    }
else:
    joblib.dump(pipe_s2_rf, MODEL_DIR / 'best_model.pkl')
    metadata = {'model_name': winner_name, 'feature_columns': FEATURE_COLS_STAGE2,
                'mae': float(winner_row['MAE']), 'rmsle': float(winner_row['RMSLE']),
                'all_results': results}

joblib.dump(metadata, MODEL_DIR / 'model_metadata.pkl')
print(f'Saved → {MODEL_DIR}')


### 6.2 Inference template — what the app would call

```python
import joblib
import pandas as pd

per_cuisine = joblib.load('model/food_demand/best_model_per_cuisine.pkl')
meta = joblib.load('model/food_demand/model_metadata.pkl')

def predict(row: dict) -> float:
    X = pd.DataFrame([row])[meta['feature_columns']]
    cuisine = row['cuisine']
    pipe = per_cuisine.get(cuisine, next(iter(per_cuisine.values())))  # fallback to any
    return max(0, float(pipe.predict(X)[0]))
```

The Streamlit app `app/app_food.py` (TODO) would wrap this with the editorial-terminal design system.